# A3-02 · Dual-namespace freeze (dry run unless you switch it on)

Checks that every unit of the A3 plan (`configs/final/amendments/holdout_A3.yaml`: `holdout.yaml` with `exp8` → `exp8a`)
has a complete, checksum-verified development-CV result at its pin, then prints the inventory. With `WRITE_FREEZE = True`
**and** an approved A3 record in `configs/final/amendments.yaml` (status `approved`, `approved_by` filled, committed) it
creates, once: the release record (Exp 0-6, release pin), the A2 record (Exp 8a, A2 pin) and the A3 master record with
their SHA-256. Rerunning cannot change an existing freeze. Nothing is trained and no holdout is evaluated here.

**How to use:** *Runtime → Run all*. Allow Google Drive access when asked. If Colab disconnects, open the notebook again
and *Run all* again: finished work is kept on Drive. Everything A3 writes goes to `<outputs>/FINAL_v1/amendments/A3_final/`
(plus, once, the two frozen records created by the freeze); no completed result, pin, QC decision or store is changed.

**Holdout policy.** The 13-participant holdout was inspected during earlier experiments (LX / WX suites), so it is no longer an untouched test set. In this release it is evaluated once per final configuration and reported descriptively; it is never used to select architectures, graph constructions, k, atlas resolution, QC thresholds or any other setting (selection reads development CV results only).

In [ ]:
# ===== Setup: nothing to edit. Mounts Drive, gets the A3 code, installs the requirements. =====
UPDATE_CODE = False   # leave False. True only to move the A3 code pin (<outputs>/FINAL_v1/amendments/A3_final/CODE_COMMIT.txt)
BRANCH = 'claude/a3-final-analysis'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])   # full history: the pins are ancestors
os.chdir('/content/repo')
_run([sys.executable, 'scripts/a3_setup.py'] + (['--update-code'] if UPDATE_CODE else []))
sys.path.insert(0, '/content/repo/src')

from adgraph.colab import sh as _sh

def sh(cmd, cwd='/content/repo'):  # every script runs with the file backend
    os.environ['MPLBACKEND'] = 'Agg'
    return _sh(cmd, cwd)

from adgraph.config import load_paths
PATHS = load_paths('/content/repo/configs/paths.yaml')
OUT = PATHS['processed_dir']
FINAL = OUT + '/FINAL_v1'
A3 = FINAL + '/amendments/A3_final'
print('A3 outputs on Drive:', A3)

## 1. Audit and dry run (writes nothing)

In [ ]:
sh("python scripts/a3.py audit")
sh("python scripts/a3.py freeze")

## 2. Write the freeze (default off; needs the author's approval recorded in the repository)

In [ ]:
WRITE_FREEZE = False   # True only after you reviewed the inventory above and approved amendment A3 in the repo
if WRITE_FREEZE:
    sh("python scripts/a3.py freeze --write")
else:
    print('WRITE_FREEZE is False: nothing was written')

## 3. Verify (only passes once a freeze exists)

In [ ]:
import os
if os.path.exists(A3 + '/freeze.json'):
    sh("python scripts/a3.py verify")
else:
    print('no freeze yet')